In [81]:
!pip install pandas_ta
!pip install yfinance

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 10.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstal

In [77]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np

In [78]:
ticker_list = ["AAPL", "MSFT", "GOOGL", "NVDA", "META"]

In [79]:
df_bulk = yf.download(tickers=ticker_list, start="2015-01-01", end="2023-12-31", group_by="ticker")

/tmp/ipykernel_15362/3797272726.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_bulk = yf.download(tickers=ticker_list, start="2015-01-01", end="2023-12-31", group_by="ticker")
[*********************100%***********************]  5 of 5 completed


In [80]:
df_stacked = df_bulk.stack(level=0, future_stack=True).rename_axis(['Date', 'Ticker']).reset_index()
df_stacked = df_stacked.sort_values(by=['Date', 'Ticker'])
df_stacked.columns.name = None
print(df_stacked.head(10))

        Date Ticker       Open       High        Low      Close     Volume
1 2015-01-02   AAPL  24.671151  24.682226  23.776353  24.214893  212818400
3 2015-01-02  GOOGL  26.430299  26.589101  26.196068  26.278944   26480000
0 2015-01-02   META  78.034911  78.382481  77.161010  77.905807   18177500
2 2015-01-02   MSFT  39.682628  40.328979  39.580574  39.767673   27913900
4 2015-01-02   NVDA   0.483011   0.486611   0.475333   0.483011  113680000
6 2015-01-05   AAPL  23.984551  24.064285  23.346676  23.532722  257142000
8 2015-01-05  GOOGL  26.159842  26.201527  25.693367  25.778225   41182000
5 2015-01-05   META  77.439062  78.700249  76.326828  76.654541   26452200
7 2015-01-05   MSFT  39.435997  39.742165  39.333943  39.401981   39673900
9 2015-01-05   NVDA   0.483011   0.484451   0.472694   0.474853  197952000


In [23]:
def fetch_technical_indicators(api_key, symbol="AAPL", interval="daily", time_period=14, series_type="close", matype=1):
    """
    Fetches PPO, OBV, ATR, and ADX data from Alpha Vantage
    and merges them into a single, chronologically sorted DataFrame.
    """
    # Base URL for all requests
    base_url = f"https://www.alphavantage.co/query?symbol={symbol}&interval={interval}&apikey={api_key}&function="

    # Dictionary of indicator endpoints (VWAP removed)
    endpoints = {
        'PPO': f"{base_url}PPO&series_type={series_type}&matype={matype}",
        'OBV': f"{base_url}OBV",
        'ATR': f"{base_url}ATR&time_period={time_period}",
        'ADX': f"{base_url}ADX&time_period={time_period}"
    }

    dataframes = []

    # Loop through each indicator and fetch the data
    for indicator, url in endpoints.items():
        print(f"Fetching {indicator} data for {symbol}...")
        response = requests.get(url).json()

        # Basic Error Checking
        if "Error Message" in response:
            print(f"API Error for {indicator}: Please check your parameters.")
            continue
        if "Information" in response:
            print(f"API Rate Limit Reached for {indicator}.")
            continue

        try:
            # The actual data is typically stored in the second key of the JSON response
            data_key = list(response.keys())[1]

            # Parse Data into a DataFrame
            df = pd.DataFrame.from_dict(response[data_key], orient='index')
            df.columns = [indicator]
            df[indicator] = df[indicator].astype(float)

            dataframes.append(df)

        except Exception as e:
            print(f"Error parsing data for {indicator}: {e}")

    # If no data was fetched successfully, exit the function
    if not dataframes:
        print("No data was successfully fetched.")
        return None

    # Merge all individual DataFrames into one
    df_combined = dataframes[0]
    for df in dataframes[1:]:
        df_combined = df_combined.join(df, how='outer')

    # Convert the index to datetime objects and sort chronologically (oldest to newest)
    df_combined.index = pd.to_datetime(df_combined.index)
    df_combined = df_combined.sort_index()

    return df_combined

In [24]:
MY_API_KEY = "AMHN58QZ560HN06D"
aapl_df = fetch_technical_indicators(MY_API_KEY)

Fetching PPO data for AAPL...
Fetching OBV data for AAPL...
API Rate Limit Reached. Free tiers are limited to 25 requests/day.
